# Tiger-DPO-RecSys — Colab 训练 notebook

完整流水线：**RQ-VAE 语义 ID → TIGER (T5-small) 生成式推荐 → 多项目 SFT → DPO 偏好对齐 → 离线评测 → 报告渲染**。

## 使用步骤

1. 把 runtime 切到 GPU：`Runtime → Change runtime type → T4 GPU` (Free) 或 `A100/V100/L4` (Pro)。
2. 配置好下面第一个 cell 的几个变量（仓库 URL、预设、Drive 备份路径）。
3. `Runtime → Run all`。耐心等。

## 三档预设

| Preset | 适合 | 数据规模 | 时长 (T4) |
| --- | --- | --- | --- |
| `local_smoke` | 验证代码不出错 | 5k 用户 | ~30 分钟 |
| `free_colab_safe` | Free Colab T4 (16 GB) | 150k 用户 | ~3-4 小时 |
| `pro_colab_full` | Colab Pro (T4/V100/A100) | 全量 ml-32m | ~8-10 小时 |

## 训练完成后

Notebook 会自动：
- 把 `models/`、`outputs/`、`logs/` 备份到 Google Drive
- 渲染 `outputs/REPORT.md`，里面包含 **TIGER+DPO vs TIGER(SFT) vs ItemKNN vs Popular vs Random** 的完整 Recall/NDCG 对比表

## 0. 配置（按需修改）

In [ ]:
# --- 项目代码来源 ---
GITHUB_URL = 'https://github.com/MasterpieceXu/Tiger-dpo-recsys.git'
GIT_BRANCH = 'main'

# --- 训练预设 ---
# 选项: 'local_smoke' | 'free_colab_safe' | 'pro_colab_full'
PRESET = 'pro_colab_full'

# 想跑哪些 stage （6 是渲染 REPORT.md，永远应该最后跑）
STAGES = '0,1,2,3,4,5,6'

# --- Google Drive 备份 ---
PERSIST_TO_DRIVE = True
DRIVE_PERSIST_PATH = '/content/drive/MyDrive/tiger-dpo-recsys-runs'

# 工作区根目录
PROJECT_DIR = '/content/Tiger-dpo-recsys'

## 1. 自检：GPU / 显存 / 磁盘 / 会话剩余时间

In [ ]:
import os, shutil, subprocess

# --- GPU 自检 ---
try:
    import torch
    assert torch.cuda.is_available(), '没有可用的 CUDA！请把 runtime 切到 GPU 后重试。'
    name = torch.cuda.get_device_name(0)
    gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'[ok] GPU: {name} ({gb:.1f} GB)')
    if gb < 14 and PRESET == 'pro_colab_full':
        print('[!] 显存 < 14 GB，pro_colab_full 预设可能 OOM。建议改成 free_colab_safe。')
except Exception as e:
    print(f'[X] GPU 自检失败: {e}')

# --- 磁盘自检 ---
free_gb = shutil.disk_usage('/content').free / 1024**3
print(f'[ok] /content 空闲磁盘: {free_gb:.1f} GB （需要至少 5 GB）')
if free_gb < 5:
    print('[!] 磁盘空间不足 5 GB，可能装不下数据集 + 检查点。')

# --- 选预设 ---
valid = {'local_smoke', 'free_colab_safe', 'pro_colab_full'}
assert PRESET in valid, f'PRESET 必须是 {valid} 之一'
print(f'[ok] preset = {PRESET}')

## 2. 克隆代码 + 挂载 Drive

In [ ]:
import os, subprocess

if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('[ok] Drive mounted')

if not os.path.exists(PROJECT_DIR):
    subprocess.check_call(['git', 'clone', '--branch', GIT_BRANCH, GITHUB_URL, PROJECT_DIR])
else:
    print(f'[ok] {PROJECT_DIR} 已存在，跳过 clone')

%cd $PROJECT_DIR
!ls

## 3. 安装依赖

Colab 自带 torch + transformers，pip 看到 requirements 里的版本约束已经满足时不会重装。第一次大约 30~60 秒。

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch, transformers, platform
print('python      :', platform.python_version())
print('torch       :', torch.__version__, '|', torch.cuda.get_device_name(0))
print('transformers:', transformers.__version__)

## 4. 下载 MovieLens-32M

约 240 MB，解压 ~1 GB。已经下过的话会自动跳过。

In [ ]:
import os
DATA_DIR = 'dataset'
TARGET = os.path.join(DATA_DIR, 'ml-32m')
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(TARGET, 'ratings.csv')):
    if not os.path.exists('ml-32m.zip'):
        !wget -q --show-progress https://files.grouplens.org/datasets/movielens/ml-32m.zip
    !unzip -q -o ml-32m.zip -d $DATA_DIR
    print('[ok] 解压完成')
else:
    print('[ok] 检测到已有 ratings.csv，跳过下载')

!ls -lh $TARGET

## 5. 跑 pipeline

完整 stages：
- 0：环境自检
- 1：数据预处理 + RQ-VAE 训练 + 语义 ID 生成
- 2：用户序列生成
- 3：TIGER 监督微调（SFT）
- 4：评测（TIGER vs Popular vs ItemKNN vs Random，自动 OOM-safe sparse baselines）
- 5：OneRec-lite 多项目 SFT + DPO 偏好对齐
- 6：再跑一次 stage 4 + 渲染 `REPORT.md`（也可以单独 `--stages 6`）

Stage 5 跑完后，stage 4 会自动既评测 SFT 模型也评测 DPO 模型，stage 6 渲染对比表。

In [ ]:
import os
os.environ['GR_PRESET'] = PRESET  # config.Config.__post_init__ 会读这个
!python scripts/run_pipeline.py --preset $PRESET --stages $STAGES

## 6. 渲染并展示 REPORT.md（如果上面跑了 stage 6 这一步是冗余的，但展示方便）

In [ ]:
import os, sys
sys.path.insert(0, PROJECT_DIR)
from src.report import generate_report

md = generate_report(
    eval_path='outputs/evaluation_results.json',
    dpo_path='outputs/dpo_metrics.json',
    output_path='outputs/REPORT.md',
    preset=PRESET,
    extras={
        'GPU': __import__('torch').cuda.get_device_name(0),
    },
)

from IPython.display import Markdown
Markdown(md)

## 7. 备份产物到 Drive

把 `models/`、`outputs/`、`logs/` 拷贝到 Google Drive 的带时间戳目录里，避免 Colab 实例回收后丢权重。

In [ ]:
import os, shutil, datetime

if PERSIST_TO_DRIVE:
    stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
    target = os.path.join(DRIVE_PERSIST_PATH, f'{PRESET}-{stamp}')
    os.makedirs(target, exist_ok=True)
    for sub in ('models', 'outputs', 'logs'):
        src = os.path.join(PROJECT_DIR, sub)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join(target, sub), dirs_exist_ok=True)
    print(f'[ok] 已备份到: {target}')
else:
    print('PERSIST_TO_DRIVE=False，跳过备份')

## 8. 试用一下推荐结果（可选）

In [ ]:
import os, sys, torch
sys.path.insert(0, PROJECT_DIR)
from src.tiger_model import TIGERModel

candidate_paths = [
    os.path.join(PROJECT_DIR, 'models', 'onerec_lite_dpo'),
    os.path.join(PROJECT_DIR, 'models', 'tiger_final'),
]
model_path = next((p for p in candidate_paths if os.path.isdir(p)), None)

if model_path is None:
    print('[!] 找不到训练好的模型，可能 stage 3/5 没跑成功')
else:
    print(f'Using {model_path}')
    m = TIGERModel.from_pretrained(model_path).to('cuda').eval()
    sample_seq = ['<id_1>', '<id_42>', '<id_7>', '<id_19>']
    print('Top-5 candidates:', m.recommend(sample_seq, num_recommendations=5, num_beams=10))